# Late Order Recovery: Agentic Supply-Chain Workflows on the NVIDIA Stack

**Workshop notebook** -- a single, end-to-end example of a model-driven agent
recovering a late customer shipment using structured tool calls, explicit skills,
and sequence-sensitive evaluation.

---
## 1. Introduction

### What this notebook demonstrates

This notebook walks through a **multi-turn agentic workflow** for late-order
recovery in a supply-chain setting. Rather than building a general-purpose agent
platform, we focus on one rich scenario and show concrete patterns for:

- **Structured tool calling** using a Nemotron-style JSON schema
- **Explicit higher-level skills** composed from deterministic tools
- **Fallback parsing** for malformed or partially structured model outputs
- **Sequence-sensitive evaluation** that checks tool-call ordering, not just outcomes
- **Training-oriented patterns** aligned with NeMo RL, Megatron, and ProRL workflows

### Why supply chain?

Supply-chain operations are a strong domain for agentic workflows because:

1. **Decisions depend on ordered information gathering.** You cannot score a
   transfer option without first knowing which alternate DC has stock, and you
   cannot check alternate stock without first confirming the primary source is
   short. This creates natural, machine-checkable sequence dependencies.

2. **Tools are deterministic but composition is not.** Each lookup (inventory,
   ETA, capacity) is a pure function, but the agent must decide *which* tools
   to call and in *what order* based on intermediate results.

3. **Mitigations involve tradeoffs.** Expediting from a supplier is faster but
   costlier; transferring from another DC is cheaper but slower. The agent
   must gather enough data to compare options before recommending.

### Why sequence correctness matters

Most agent evaluations check whether the final answer is correct. But in
supply-chain recovery, *how* the agent arrived at the answer matters:

- Calling `get_transfer_eta` before `find_alternate_inventory` is logically
  invalid -- you cannot estimate a transfer without knowing the source.
- Calling `score_recovery_options` before gathering any options is pointless.
- An agent that guesses the right answer without proper investigation would
  be unsafe in production.

We define and enforce these dependencies explicitly, making the evaluation
**sequence-sensitive** rather than outcome-only.

---
## 2. Scenario Definition

### Business problem

Customer order **SO-10482** for **1,200 units** of **SKU-4090** is at risk of
missing its committed delivery date. The order was placed against distribution
center **DC-WEST-01**, which is the primary source. Shipment tracking shows
the order has not yet shipped and the committed date is approaching.

### Target task

> Determine whether order `SO-10482` can still be fulfilled on time from its
> primary source. If not, recommend the best mitigation action from the
> available options: fulfill from original DC, transfer from an alternate DC,
> expedite from a supplier, partially fulfill, or substitute with a compatible
> SKU.

### Success criteria

A successful agent run must:

1. **Diagnose the risk** -- confirm the order exists, retrieve shipment status,
   and identify that the order is at risk.
2. **Check primary fulfillment** -- verify inventory and capacity at DC-WEST-01.
3. **Explore alternates** -- check at least one alternate DC or supplier path.
4. **Score and recommend** -- compare candidate options and produce a final
   recommendation with rationale.
5. **Respect sequence dependencies** -- never call a downstream tool before its
   prerequisites.
6. **Complete in 5-10 tool calls** -- efficient but thorough.

### Sequence dependencies (machine-checkable)

```
get_order
  -> get_shipment_status
  -> get_inventory(source DC)
  -> get_fulfillment_capacity(source DC)
  -> get_supplier_expedite_options

get_inventory(source DC)
  -> find_alternate_inventory
     -> get_transfer_eta

at least one mitigation path explored
  -> score_recovery_options
     -> recommend_action
```

### Mitigation options the agent should evaluate

| Option | Description |
|--------|-------------|
| Primary fulfill | Ship from DC-WEST-01 if enough stock and capacity |
| DC transfer | Transfer from an alternate DC with available inventory |
| Supplier expedite | Rush order directly from a supplier |
| Partial fulfillment | Ship what is available, backorder the rest |
| SKU substitute | Use a compatible alternate SKU if enabled |

---
## 3. Synthetic Data Model

All scenario data lives in `src/scenario_data.py` as plain Python dicts.
The data is small enough to print in full and inspect cell-by-cell.

Tables:
- **ORDERS** -- customer orders (keyed by order_id)
- **SHIPMENTS** -- shipment status per order
- **INVENTORY** -- on-hand inventory by (sku, dc_id)
- **TRANSFER_LANES** -- routes between DCs with lead times
- **SUPPLIER_OPTIONS** -- supplier expedite options by sku
- **FULFILLMENT_CAPACITY** -- available capacity by (dc_id, date)
- **SUBSTITUTE_SKUS** -- optional substitute SKU mappings

In [ ]:
import sys, pathlib
# Ensure src/ is importable from the notebook
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
# Also allow running from repo root
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [ ]:
from src.scenario_data import (
    ORDERS, SHIPMENTS, INVENTORY, TRANSFER_LANES,
    SUPPLIER_OPTIONS, FULFILLMENT_CAPACITY, SUBSTITUTE_SKUS,
)

# Phase 2 will populate these tables. Quick sanity check:
print(f"Orders:     {len(ORDERS)}")
print(f"Shipments:  {len(SHIPMENTS)}")
print(f"Inventory:  {len(INVENTORY)}")
print(f"Transfers:  {len(TRANSFER_LANES)}")
print(f"Suppliers:  {len(SUPPLIER_OPTIONS)}")
print(f"Capacity:   {len(FULFILLMENT_CAPACITY)}")
print(f"Substitutes:{len(SUBSTITUTE_SKUS)}")

---
## 4. Skill Definitions

Skills are **higher-level behaviors** composed from deterministic tools.
The agent selects a skill; the skill defines which tools to call and in what
order. This separation makes it possible to evaluate *skill selection* and
*tool use* independently.

| Skill | Purpose | Tools used |
|-------|---------|------------|
| `diagnose_order_risk` | Understand current order/shipment state | `get_order`, `get_shipment_status` |
| `assess_primary_fulfillment` | Check if source DC can still fulfill | `get_inventory`, `get_fulfillment_capacity` |
| `evaluate_alternate_recovery_paths` | Explore alternate DCs and suppliers | `find_alternate_inventory`, `get_transfer_eta`, `get_supplier_expedite_options` |
| `synthesize_recommendation` | Score options and recommend action | `score_recovery_options`, `recommend_action` |

### Skills vs. tools

- A **tool** is a single deterministic lookup or computation.
- A **skill** is a reusable behavior pattern that orchestrates multiple tools.
- The agent reasons at the **skill level** ("I should diagnose the order risk
  first") and the execution engine translates that into **tool-level** calls.

This two-layer design lets us:
1. Evaluate whether the agent picked the right skill for the situation.
2. Evaluate whether the tools within each skill were called correctly.
3. Supervise at the skill level for training (coarser, more robust signal).

In [ ]:
from src.skills import SKILL_NAMES, SKILL_TOOL_PATTERNS

for skill in SKILL_NAMES:
    tools = SKILL_TOOL_PATTERNS[skill]
    print(f"{skill}:")
    for t in tools:
        print(f"  -> {t}")
    print()

---
## 5. Tool Definitions

Each tool is a pure function over the synthetic data tables. Tools are
registered in `TOOL_REGISTRY` so the agent loop can dispatch by name.

| Tool | Inputs | Output | Dependencies |
|------|--------|--------|-------------|
| `get_order` | order_id | Order details | (none) |
| `get_shipment_status` | order_id | Shipment state | get_order |
| `get_inventory` | sku, dc_id | Stock levels | get_order |
| `find_alternate_inventory` | sku, region | Alternate DC stock | get_inventory |
| `get_transfer_eta` | from_dc, to_dc, sku, qty | Transfer estimate | find_alternate_inventory |
| `get_supplier_expedite_options` | sku, qty | Supplier rush options | get_order |
| `get_fulfillment_capacity` | dc_id, date | DC capacity | get_order |
| `score_recovery_options` | options, objective | Ranked options | >= 1 mitigation path |
| `recommend_action` | context | Final recommendation | score_recovery_options |

In [ ]:
from src.tools import TOOL_REGISTRY, TOOL_DEPENDENCIES

print("Registered tools:", len(TOOL_REGISTRY))
print()
print("Dependency graph:")
for tool, deps in TOOL_DEPENDENCIES.items():
    if deps:
        print(f"  {tool} <- {deps}")
    else:
        print(f"  {tool} <- (none)")

---
## 6. Structured Tool-Call Schema

The model must emit tool calls in a canonical **Nemotron-style JSON format**.
This section defines the schema and shows valid and invalid examples.

### Canonical format

```json
{
  "thought": "optional short reasoning summary",
  "tool_call": {
    "name": "get_inventory",
    "arguments": {
      "sku": "SKU-4090",
      "dc_id": "DC-WEST-01"
    }
  }
}
```

### Valid examples

```json
{"tool_call": {"name": "get_order", "arguments": {"order_id": "SO-10482"}}}
```

```json
{
  "thought": "Order is at risk, check if source DC has stock.",
  "tool_call": {
    "name": "get_inventory",
    "arguments": {"sku": "SKU-4090", "dc_id": "DC-WEST-01"}
  }
}
```

### Invalid examples

Missing `tool_call` key:
```json
{"name": "get_order", "arguments": {"order_id": "SO-10482"}}
```

Unknown tool:
```json
{"tool_call": {"name": "query_database", "arguments": {"sql": "SELECT *"}}}
```

Missing arguments:
```json
{"tool_call": {"name": "get_inventory", "arguments": {}}}
```

In [ ]:
from src.schema import ParsedToolCall, ValidationError, REQUIRED_KEYS, OPTIONAL_KEYS

print("Schema constants:")
print(f"  Required top-level keys: {REQUIRED_KEYS}")
print(f"  Optional top-level keys: {OPTIONAL_KEYS}")

---
## 7. Agent Loop

The agent loop follows the **OpenCode** execution model:

```
think -> emit tool call -> validate -> execute -> observe -> continue
```

Key design choices:

- **Bounded iterations**: hard stop at 15 iterations to prevent runaway loops.
- **Skill awareness**: the system prompt tells the model which skills are
  available and what tool sequences each skill expects.
- **Validation before execution**: every tool call is checked against the
  schema and dependency graph *before* the tool runs.
- **Full trace capture**: every step is recorded in an `AgentTrace` so the
  notebook can replay and evaluate the trajectory afterward.
- **Single model adapter**: the only network call is `call_model()` in
  `src/agent_loop.py`. Everything else is local and deterministic.

### Stop conditions

The loop terminates when:
1. The model emits a **final answer** (no `tool_call`, just a recommendation).
2. The model has executed `recommend_action` and returned a result.
3. **Max iterations** reached (safety bound).
4. An **unrecoverable error** occurs.

In [ ]:
from src.agent_loop import (
    MODEL_ENDPOINT, MODEL_NAME, MAX_ITERATIONS,
    AgentTrace, ToolCallRecord,
)

print(f"Model endpoint:  {MODEL_ENDPOINT}")
print(f"Model name:      {MODEL_NAME}")
print(f"Max iterations:  {MAX_ITERATIONS}")

---
## 8. Fallback Parsing

Real models produce messy outputs. This section defines the **repair-vs-reject
policy** for common failure modes.

| Failure mode | Policy | Rationale |
|--------------|--------|-----------|
| Malformed JSON (minor) | **Repair** | Fix trailing commas, missing quotes |
| Mixed text + JSON | **Repair** | Extract JSON block from surrounding text |
| Missing optional fields | **Repair** | Fill defaults (e.g., `thought: null`) |
| Missing `tool_call` | **Reject** | Intent is ambiguous |
| Unknown tool name | **Reject** | Could mean anything |
| Unsafe arguments | **Reject** | Cannot guess safe values |

The fallback layer sits between the model output parser and the tool
dispatcher. It returns a `FallbackResult` that records what was attempted
and whether the output was repaired or rejected.

In [ ]:
from src.fallbacks import FallbackAction, FallbackResult

print("Fallback actions:", [a.value for a in FallbackAction])

---
## 9. Worked Examples

This section will contain two complete trajectories:

### 9a. Successful trajectory

A clean run where the agent:
1. Diagnoses the order risk (get_order -> get_shipment_status)
2. Checks primary fulfillment (get_inventory -> get_fulfillment_capacity)
3. Explores alternates (find_alternate_inventory -> get_transfer_eta,
   get_supplier_expedite_options)
4. Scores and recommends (score_recovery_options -> recommend_action)

Expected: ~8 tool calls, all valid, correct sequence, good recommendation.

### 9b. Failure / repair trajectory

A run where the model produces at least one malformed output that triggers
the fallback parser. Shows the repair-vs-reject decision and how the agent
recovers and continues.

*(Phase 6 will populate these with real traces.)*

In [ ]:
# Placeholder for worked example execution.
# Phase 4-6 will wire this up with real model calls and trace replay.
print("Worked examples will be populated in Phases 4-6.")

---
## 10. Evaluation

Trajectories are evaluated across **seven dimensions**:

| Dimension | What it measures | Weight |
|-----------|-----------------|--------|
| Skill selection | Did the agent pick the right skills in order? | High |
| Tool validity | Were all tool calls well-formed JSON? | High |
| Tool accuracy | Were tool arguments correct for the scenario? | Medium |
| Sequence correctness | Were dependency constraints respected? | **Critical** |
| Task success | Did the agent reach a valid recommendation? | High |
| Recovery quality | If fallback was needed, was repair correct? | Medium |
| Efficiency | Steps taken vs. optimal trajectory length | Low |

**Sequence correctness** is the distinguishing evaluator for this workshop.
It checks the tool-call order against the dependency graph, not just the
final answer.

In [ ]:
from src.evaluation import EVAL_DIMENSIONS

print("Evaluation dimensions:")
for dim in EVAL_DIMENSIONS:
    print(f"  - {dim}")

---
## 11. Training-Oriented Discussion

This section connects the workshop patterns to the NVIDIA training stack.

### What should be supervised?

- **Skill level**: which skill the agent selected at each decision point.
  Coarser signal, more robust for training.
- **Trajectory level**: the full sequence of tool calls. Finer signal, but
  noisier because multiple valid orderings may exist.

### How trajectories could be scored

Using the seven evaluation dimensions above as reward components, weighted
by business priority. Sequence correctness should carry the highest weight
to avoid rewarding agents that guess correct answers via invalid paths.

### Sequence-sensitive rewards (ProRL framing)

**NVIDIA ProRL** provides a framework for designing rewards around:
- Tool validity (binary: was the call well-formed?)
- Sequence correctness (dependency graph compliance)
- Recovery quality (did the agent recover from malformed outputs?)
- Task success (did the final recommendation meet criteria?)

### Mapping to NeMo RL / Megatron workflows

- **NeMo Data Designer**: author and iterate on synthetic traces, repair
  cases, and evaluation examples for this scenario.
- **NeMo RL**: export trajectories with per-step rewards for RL fine-tuning.
- **NVIDIA Megatron**: scale model training on 8x H100 GPUs using the
  trajectory data and reward signals produced above.
- **OpenCode**: the agent loop harness that produced the traces, providing
  the execution and evaluation infrastructure.

### NeMo Guardrails (optional follow-up)

If the fallback parsing layer is extended to include policy checks (e.g.,
rejecting tool calls with unsafe arguments, enforcing business rules),
**NeMo Guardrails** would be the natural integration point. This is noted
as an optional follow-up, not core MVP scope.

In [ ]:
# Placeholder for training-oriented export examples.
# Phase 7 will add concrete NeMo RL export and ProRL reward examples.
print("Training-oriented examples will be populated in Phase 7.")

---
## Next steps

This skeleton establishes the scenario, section structure, and module
boundaries. The remaining phases will populate each section:

- **Phase 2**: Synthetic data tables and deterministic tool implementations
- **Phase 3**: Skill implementations with tool orchestration
- **Phase 4**: Schema validation, model adapter, and agent loop
- **Phase 5**: Fallback parsing and recovery examples
- **Phase 6**: Worked traces and evaluation
- **Phase 7**: Training-oriented discussion with NeMo RL / Megatron examples